# SCEDG: main pipeline

Semantic-compressive evidence selection and dynamic gating for knowledge-based visual question answering, with Qwen2.5-VL-3B as the answer generator.

Run [`Preparation.ipynb`](Preparation.ipynb) once to build the LongCLIP-L FAISS index over Wiki6M. Then set `DATASET_NAME` and `DATASET_SPLIT` in **3. Configuration** and run this notebook from top to bottom, once per split:

| Step | `DATASET_SPLIT` | What the notebook does |
|---|---|---|
| 1 | `train` | multi-view retrieval, cross-encoder, SCE, task-specific reranker training, Qwen2.5-VL LoRA fine-tuning |
| 2 | `validation` | the same evidence pipeline, then evaluation of every saved checkpoint for gate thresholds 0.55 to 1.0 |
| 3 | `test` | the same evidence pipeline, then evaluation of the final adapter with gate threshold 0.9 |

For OK-VQA, the official training split is partitioned into `train` (85%) and `validation` (15%) by image ID with seed 42, so questions about the same image never cross splits. The official OK-VQA test split is kept unchanged. A-OKVQA uses its native `train`, `validation`, and `test` splits. A-OKVQA test answers are hidden, so test scores must be obtained from the official evaluation server.

The multi-view retrieval stage follows Section 3.2 of the paper. REACT builds the scene graph, its textual triples form the contextual caption, GLiNER + spaCy concepts are aligned to graph entities for local crops, and graph entities are aligned to ConceptNet with MiniLM for entity anchors. Each query view is retrieved independently and fused with RRF.

REACT uses the `Maelic/SGG-Benchmark` implementation with `REACTPredictor` and the native `yoloworld` backbone (`yolov8x-worldv2`). The notebook selects a relation checkpoint from `REACT_RUN_DIR` only when it was trained with this configuration. Do not mix a REACT/REACT++ checkpoint trained with a different YOLO backbone.

Large intermediate files are stored under `/content/kg_vqa_cache/<dataset>/<split>`. Model checkpoints and experiment logs use the paths defined in **3. Configuration**.

**Before running**
1. Mount Google Drive and set `ROOT_DIR`.
2. Add `HF_TOKEN` (with write access) to Colab Secrets and set `HF_USERNAME` to your Hugging Face account.
3. Run `Preparation.ipynb` and check that `CHUNKS_DIR` contains matching `.index` and metadata `.json` files.
4. Place a REACT + YOLO-WorldV2 relation checkpoint in `REACT_RUN_DIR` if the scene-graph cache has not been generated yet.

# 1. Drive Mount


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


# 2. Environment Settings


In [ ]:
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --default-timeout=10000
!pip install https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --default-timeout=10000
!pip install faiss-gpu-cu12 datasets transformers==5.1.0
!pip install -U accelerate sentence-transformers bitsandbytes peft torchao
!pip install ftfy regex qwen-vl-utils gliner spacy requests scikit-learn
!python -m spacy download en_core_web_trf --default-timeout=10000
!pip uninstall -y torchcodec


In [ ]:
import os
import subprocess
import sys

if not os.path.exists("Long-CLIP"):
    subprocess.run(
        ["git", "clone", "https://github.com/beichenzbc/Long-CLIP.git"],
        check=True,
    )

longclip_path = os.path.abspath("Long-CLIP")
if longclip_path not in sys.path:
    sys.path.append(longclip_path)

REACT_REPO_DIR_BOOTSTRAP = "/content/SGG-Benchmark"
if not os.path.exists(REACT_REPO_DIR_BOOTSTRAP):
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Maelic/SGG-Benchmark.git",
            REACT_REPO_DIR_BOOTSTRAP,
        ],
        check=True,
    )

react_python = os.path.join(REACT_REPO_DIR_BOOTSTRAP, ".venv", "bin", "python")
if not os.path.exists(react_python):
    subprocess.run(
        ["bash", "scripts/install_uv.sh"],
        cwd=REACT_REPO_DIR_BOOTSTRAP,
        check=True,
    )

react_vg_annotations = os.path.join(
    REACT_REPO_DIR_BOOTSTRAP,
    "datasets", "VG150", "VG150_coco_format", "train", "_annotations.coco.json",
)
if not os.path.exists(react_vg_annotations):
    subprocess.run(
        [react_python, "tools/download_from_hub.py", "--dataset", "VG150"],
        cwd=REACT_REPO_DIR_BOOTSTRAP,
        check=True,
    )


In [ ]:
import ast
import gc
import glob
import gzip
import hashlib
import json
import math
import os
import random
import re
import string
import subprocess
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from typing import Any
from urllib.parse import unquote

import faiss
import numpy as np
import requests
import spacy
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from gliner import GLiNER
from huggingface_hub import HfApi, login, snapshot_download
from PIL import Image
from peft import LoraConfig, PeftModel, get_peft_model
from sentence_transformers import CrossEncoder, SentenceTransformer
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoProcessor,
    AutoTokenizer,
    CLIPModel,
    CLIPProcessor,
    CLIPTokenizer,
    Qwen2_5_VLForConditionalGeneration,
    Trainer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    TrainingArguments,
    set_seed,
)
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel
from qwen_vl_utils import process_vision_info


# 3. Configuration


In [ ]:
ROOT_DIR = "/content/drive/MyDrive/dataset"
DATASET_NAME = "OK-VQA"
DATASET_SPLIT = "train"

SEED = 42
OKVQA_VAL_RATIO = 0.15
HF_USERNAME = "your-hf-username"  # TODO: your Hugging Face account; models are pushed to <HF_USERNAME>/<RUN_NAME>

DATASET_NAME = DATASET_NAME.upper().strip()
DATASET_SPLIT = DATASET_SPLIT.lower().strip()
globals().pop("_DATASET_CACHE", None)
globals().pop("_DATASET_LOOKUP_CACHE", None)

if DATASET_NAME == "OK-VQA":
    if DATASET_SPLIT in {"train", "validation"}:
        HF_DATASET_ID, HF_SPLIT = "Multimodal-Fatima/OK-VQA_train", "train"
    elif DATASET_SPLIT == "test":
        HF_DATASET_ID, HF_SPLIT = "Multimodal-Fatima/OK-VQA_test", "test"
    else:
        raise ValueError("OK-VQA supports train, validation, or test.")
    DATASET_SLUG = "okvqa"
elif DATASET_NAME == "A-OKVQA":
    if DATASET_SPLIT not in {"train", "validation", "test"}:
        raise ValueError("A-OKVQA supports train, validation, or test.")
    HF_DATASET_ID, HF_SPLIT = "HuggingFaceM4/A-OKVQA", DATASET_SPLIT
    DATASET_SLUG = "aokvqa"
else:
    raise ValueError("DATASET_NAME must be OK-VQA or A-OKVQA.")

CACHE_DIR = os.path.join("/content/kg_vqa_cache", DATASET_SLUG, DATASET_SPLIT)
os.makedirs(CACHE_DIR, exist_ok=True)

MKG_DIR = os.path.join(ROOT_DIR, "KB")
CHUNKS_DIR = os.path.join(MKG_DIR, "Chunk_files", "Long-CLIP-L_Wiki6M")
WIKI6M_PATH = os.path.join(MKG_DIR, "Wiki6M_ver_1_0.jsonl.gz")

RAW_KNOWLEDGE_FILE = os.path.join(CACHE_DIR, "raw_retrieval.jsonl")
RERANKED_TOPM_FILE = os.path.join(CACHE_DIR, "cross_encoder_top50.jsonl")
RERANKED_KNOWLEDGE = os.path.join(CACHE_DIR, "mamba_top10.jsonl")
SUMMARY_DATA = os.path.join(CACHE_DIR, "summary_dataset.json")
MXBAI_LABEL = os.path.join(CACHE_DIR, "cross_encoder_labels.json")
SUMMARY_TOP_1 = os.path.join(CACHE_DIR, "dataset_one_knowledge.json")

MODEL_DIR = os.path.join(
    ROOT_DIR, "models", DATASET_SLUG, "cross_encoder", "mixedbread"
)
RUN_NAME = f"{DATASET_SLUG}_qwen_k_6m_mambancd_rerank_run1"
MODEL_REPO_ID = f"{HF_USERNAME}/{RUN_NAME}"
TRACKING_REPO_ID = f"{HF_USERNAME}/kg-vqa_experimental_results"

BASE_LOG_DIR = "/content/experiment_logs"
TEMP_LOG_DIR = os.path.join(BASE_LOG_DIR, RUN_NAME)
TRACKER_FILE = os.path.join(TEMP_LOG_DIR, "eval_results_tracker.jsonl")
TEMP_MODEL_DIR = os.path.join("/content", f"{RUN_NAME}_model")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TEMP_LOG_DIR, exist_ok=True)
os.makedirs(TEMP_MODEL_DIR, exist_ok=True)

MODEL_TYPE = "longclip_large"
PIPELINE_BATCH_SIZE = 64
RETRIEVAL_TOP_K_PER_QUERY = 120
RRF_K = 60
RAW_FUSED_TOP_K = 350
RERANK_TOP_K = 50
MAMBA_INPUT_TOP_M = 25
FINAL_KNOWLEDGE_TOP_K = 10

GLINER_MODEL_ID = "urchade/gliner_medium-v2.1"
SPACY_MODEL_NAME = "en_core_web_trf"
MINILM_MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
CROP_PADDING = 5
MIN_CROP_SIZE = 20

REACT_REPO_DIR = "/content/SGG-Benchmark"
REACT_PYTHON = os.path.join(REACT_REPO_DIR, ".venv", "bin", "python")
REACT_CONFIG_PATH = os.path.join(CACHE_DIR, "react_yoloworldv2_hydra.yaml")
REACT_RUNNER_PATH = os.path.join(CACHE_DIR, "react_yoloworld_custom_infer.py")
REACT_RUN_DIR = os.path.join(ROOT_DIR, "models", "react_yoloworldv2_vg150")
REACT_MODEL_WEIGHT = None

REACT_YOLOWORLD_MODEL_NAME = "yolov8x-worldv2.pt"
REACT_YOLOWORLD_DETECTOR_CKPT = os.path.join(
    REACT_REPO_DIR, REACT_YOLOWORLD_MODEL_NAME
)
REACT_GLOVE_DIR = os.path.join(REACT_REPO_DIR, "datasets")
REACT_IMAGE_DIR = os.path.join(CACHE_DIR, "react_images")
REACT_OUTPUT_DIR = os.path.join(CACHE_DIR, "react_sgg")
REACT_AUTO_RUN = True
REACT_TEST_BATCH_SIZE = 1

CONCEPTNET_API = "https://api.conceptnet.io"
CONCEPTNET_CACHE_FILE = os.path.join(CACHE_DIR, "conceptnet_minilm_cache.json")
CONCEPTNET_RELATED_LIMIT = 50
CONCEPTNET_NEIGHBOR_LIMIT = 200
CONCEPTNET_STRICT = True

CROSS_ENCODER_MODEL_ID = "mixedbread-ai/mxbai-rerank-large-v1"
MAMBA_MODEL_ID = "state-spaces/mamba2-130m"
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dataset: {DATASET_NAME} / {DATASET_SPLIT}")
print(f"Hugging Face source: {HF_DATASET_ID} ({HF_SPLIT})")
print(f"Cache: {CACHE_DIR}")
print(f"Device: {device}")


In [ ]:
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
set_seed(SEED)
print(f"Random seed: {SEED}")


# 4. Dataset loading and multi-view query formation

Loads the split from the Hugging Face Hub, builds REACT scene graphs, and forms the visual, contextual, and entity-anchor query views (Section 3.2 of the paper).

In [ ]:
_DATASET_CACHE = None
_DATASET_LOOKUP_CACHE = None

def canonical_id(value):
    if hasattr(value, "item"):
        value = value.item()
    return str(value)

def parse_answer_list(value):
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            value = parsed if isinstance(parsed, (list, tuple)) else [parsed]
        except Exception:
            value = [text]
    if isinstance(value, dict):
        value = [value]
    answers = []
    for answer in value:
        if isinstance(answer, dict):
            answer = answer.get("answer", answer.get("raw_answer", ""))
        answer = str(answer).strip()
        if answer:
            answers.append(answer)
    return answers

def get_answers(item):
    if DATASET_NAME == "OK-VQA":
        answers = parse_answer_list(item.get("answers"))
        return answers or parse_answer_list(item.get("answers_original"))
    return parse_answer_list(item.get("direct_answers"))

def get_okvqa_image_id(item):
    image_id = item.get("id_image", item.get("image_id"))
    if image_id is not None:
        return canonical_id(image_id)
    question_id = item.get("question_id")
    try:
        return canonical_id(int(question_id) // 10)
    except Exception as error:
        raise KeyError("OK-VQA sample has no usable image ID.") from error

def get_sample_fields(item, include_image=True):
    question_id = item["question_id"]
    if DATASET_NAME == "OK-VQA":
        image_id = get_okvqa_image_id(item)
    else:
        image_id = item.get("image_id", question_id)
    sample = {
        "question_id": question_id,
        "image_id": image_id,
        "question": str(item["question"]).strip(),
        "answers": get_answers(item),
    }
    if include_image:
        sample["image"] = item["image"].convert("RGB")
    return sample

def split_okvqa_train_validation(dataset):
    if "id_image" in dataset.column_names:
        image_ids = [canonical_id(value) for value in dataset["id_image"]]
    elif "image_id" in dataset.column_names:
        image_ids = [canonical_id(value) for value in dataset["image_id"]]
    else:
        image_ids = [canonical_id(int(value) // 10) for value in dataset["question_id"]]
    unique_image_ids = list(dict.fromkeys(image_ids))
    train_ids, validation_ids = train_test_split(
        unique_image_ids,
        test_size=OKVQA_VAL_RATIO,
        random_state=SEED,
        shuffle=True,
    )
    selected_ids = set(
        train_ids if DATASET_SPLIT == "train" else validation_ids
    )
    indices = [
        index for index, image_id in enumerate(image_ids)
        if image_id in selected_ids
    ]
    return dataset.select(indices), len(train_ids), len(validation_ids)

def load_vqa_dataset(force_reload=False):
    global _DATASET_CACHE
    if _DATASET_CACHE is not None and not force_reload:
        return _DATASET_CACHE

    dataset = load_dataset(HF_DATASET_ID, split=HF_SPLIT)
    required = {"image", "question", "question_id"}
    missing = required - set(dataset.column_names)
    if missing:
        raise KeyError(
            f"Missing columns {sorted(missing)}. Available: {dataset.column_names}"
        )

    keep_columns = [
        column
        for column in (
            "image",
            "question",
            "question_id",
            "id_image",
            "image_id",
            "answers",
            "answers_original",
            "direct_answers",
        )
        if column in dataset.column_names
    ]
    dataset = dataset.select_columns(keep_columns)

    if DATASET_NAME == "OK-VQA" and DATASET_SPLIT in {"train", "validation"}:
        dataset, train_image_count, validation_image_count = (
            split_okvqa_train_validation(dataset)
        )
        print(
            "OK-VQA image split: "
            f"{train_image_count:,} train images / "
            f"{validation_image_count:,} validation images"
        )

    _DATASET_CACHE = dataset
    print(f"Loaded {len(dataset):,} samples for {DATASET_NAME}/{DATASET_SPLIT}")
    return dataset

class DatasetLookup:
    def __init__(self, dataset):
        self.dataset = dataset
        self.question_to_index = {
            canonical_id(question_id): index
            for index, question_id in enumerate(dataset["question_id"])
        }

    def get_item(self, question_id):
        return self.dataset[self.question_to_index[canonical_id(question_id)]]

    def get_image(self, question_id):
        return self.get_item(question_id)["image"].convert("RGB")

def get_dataset_lookup():
    global _DATASET_LOOKUP_CACHE
    if _DATASET_LOOKUP_CACHE is None:
        _DATASET_LOOKUP_CACHE = DatasetLookup(load_vqa_dataset())
    return _DATASET_LOOKUP_CACHE

def read_processed_ids(jsonl_path):
    processed = set()
    if not os.path.exists(jsonl_path):
        return processed
    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                processed.add(canonical_id(json.loads(line)["question_id"]))
            except Exception:
                continue
    return processed

def release_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
TARGET_LABELS = [
    "person", "people", "group of people", "animal", "creature",
    "physical object", "item", "tool", "vehicle", "transportation",
    "food", "drink", "dessert", "furniture", "appliance", "location",
    "place", "scenery", "room", "body part", "clothing", "accessory",
    "plant", "flower", "tree", "vegetable", "organization", "brand",
    "named entity", "product", "building", "sports equipment"
]

BLACKLIST_EXACT = {
    "what", "which", "who", "whom", "whose", "where", "when", "why", "how",
    "picture", "image", "photo", "scene", "view", "part", "name", "object",
    "objects", "thing", "things", "item", "items", "stuff", "kind", "type",
    "sort", "form", "style", "way", "this", "that", "these", "those", "it",
    "its", "they", "them", "you", "your", "we", "us", "he", "she", "him",
    "her", "can", "do", "does", "did", "is", "are", "was", "were", "be"
}

DETERMINER_RE = re.compile(
    r"^(?:the|a|an|this|that|these|those|my|your|his|her|its|our|their)\s+",
    flags=re.IGNORECASE,
)

def clean_phrase(text):
    text = re.sub(r"\s+", " ", str(text)).strip().lower()
    text = DETERMINER_RE.sub("", text)
    return text.strip(" ?!.,;:\"'")

def dedupe_keep_order(values):
    seen = set()
    output = []
    for value in values:
        value = clean_phrase(value)
        if value and value not in seen:
            seen.add(value)
            output.append(value)
    return output

def extract_question_entity_anchors(question, nlp_model):
    """N_q in the paper: noun chunks plus named entities from the question."""
    if not question:
        return []
    doc = nlp_model(question)
    values = [chunk.text for chunk in doc.noun_chunks]
    values.extend(entity.text for entity in doc.ents)
    return [
        value for value in dedupe_keep_order(values)
        if value not in BLACKLIST_EXACT and len(value) > 1
    ]

def batch_extract_visual_concepts(batch_texts, gliner_model, nlp_model):
    """GLiNER + spaCy concepts from the question, later aligned to the REACT scene graph G."""
    predictions = gliner_model.batch_predict_entities(
        batch_texts, TARGET_LABELS, threshold=0.4
    )
    docs = list(nlp_model.pipe(batch_texts))
    outputs = []
    for entities, doc in zip(predictions, docs):
        phrases = [entity["text"] for entity in entities]
        phrases.extend(chunk.text for chunk in doc.noun_chunks)
        concepts = [
            value for value in dedupe_keep_order(phrases)
            if value not in BLACKLIST_EXACT and len(value) > 1
        ]
        outputs.append(concepts)
    return outputs

def cosine_matrix(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    a /= np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)
    b /= np.maximum(np.linalg.norm(b, axis=1, keepdims=True), 1e-12)
    return a @ b.T


In [ ]:
REACT_YOLOWORLDV2_CONFIG = r"""
# @package _global_
output_dir: "./checkpoints/VG150/react_yoloworldv2"
glove_dir: "datasets/"

seed: 42
metric_to_track: "mR"
dtype: "float32"
verbose: "INFO"

input:
  img_size: [640, 640]
  pixel_mean: [102.9801, 115.9465, 122.7717]
  pixel_std: [1.0, 1.0, 1.0]
  to_bgr255: true
  flip_prob_train: 0.5
  padding: true
  brightness: 0.0
  contrast: 0.0
  saturation: 0.0
  hue: 0.0
  vertical_flip_prob_train: 0.0

datasets:
  name: "VG150"
  type: "coco"
  data_dir: "datasets/VG150/VG150_coco_format"

dataloader:
  num_workers: 4
  size_divisibility: 32
  aspect_ratio_grouping: true

model:
  relation_on: true
  device: "cuda"
  meta_architecture: "GeneralizedYOLO"
  pretrained_detector_ckpt: "yolov8x-worldv2.pt"
  text_embedding: "glove.6B"
  box_head: false

  backbone:
    type: "yoloworld"
    nms_thresh: 0.001
    freeze: true
    freeze_at: 10

  yolo:
    size: "yolov8x-worldv2"
    out_channels: [320]

  roi_box_head:
    feature_extractor: "YOLOV8FeatureExtractor"
    pooler_resolution: 7
    pooler_sampling_ratio: 2
    pooler_scales: [0.125, 0.0625, 0.03125]
    mlp_head_dim: 1024
    num_classes: 151

  roi_heads:
    fg_iou_threshold: 0.3
    bg_iou_threshold: 0.1
    nms: 0.2
    detections_per_img: 100
    nms_filter_duplicates: true

  roi_relation_head:
    predictor: "REACTPredictor"
    feature_extractor: "RelationFeatureExtractor"
    use_scene_context: false
    num_classes: 51
    embed_dim: 200
    mlp_head_dim: 1024
    context_dropout_rate: 0.2
    context_hidden_dim: 512
    context_pooling_dim: 2048
    use_union_features: false
    use_spatial_features: true
    use_union_features_inference: false
    batch_size_per_image: 256
    positive_fraction: 0.25
    num_sample_per_gt_rel: 4
    require_box_overlap: false
    add_gtbox_to_proposal_in_train: true
    use_gt_box: false
    use_gt_object_label: false
    loss:
      loss_type: "CrossEntropyLoss"
    use_frequency_bias: false
    react_loss_weights:
      l21_loss: 1.0
      dist_loss2: 0.1
      loss_dis: 0.5

solver:
  max_epoch: 20
  base_lr: 1.0e-04
  bias_lr_factor: 1
  momentum: 0.9
  weight_decay: 0.05
  weight_decay_bias: 0.0
  gamma: 0.5
  warmup_factor: 0.1
  warmup_epochs: 1
  warmup_method: "linear"
  checkpoint_period: 500
  grad_norm_clip: 5.0
  accum_steps: 1
  print_grad_freq: 250
  pre_val: true
  val_period: 500
  ims_per_batch: 8
  optimizer: "ADAMW"
  schedule:
    type: "WarmupCosineAnnealingIterLR"
    eta_min: 1.0e-6

test:
  ims_per_batch: 1
  detections_per_img: 100
  informative: false
  bbox_aug:
    enabled: false
    h_flip: false
    scales: []
    max_size: 4000
    scale_h_flip: false
  save_proposals: false
  relation:
    multiple_preds: false
    iou_threshold: 0.5
    require_overlap: true
    later_nms_prediction_thres: 0.5
    sync_gather: true
  allow_load_from_cache: false
  top_k: 100
""".strip()

REACT_CURRENT_MAIN_RUNNER = r"""
import argparse
import json
import math
import os
import shutil
import sys
from pathlib import Path

import cv2
import numpy as np
import torch


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--repo", required=True)
    p.add_argument("--config", required=True)
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--detector", required=True)
    p.add_argument("--detector-name", default="yolov8x-worldv2.pt")
    p.add_argument("--images", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--batch-size", type=int, default=1)
    return p.parse_args()


args = parse_args()
sys.path.insert(0, os.path.abspath(args.repo))
os.chdir(args.repo)

from omegaconf import open_dict
from ultralytics.utils.downloads import attempt_download_asset
from sgg_benchmark.config.hydra_config import load_config_from_file
from sgg_benchmark.data import get_dataset_statistics
from sgg_benchmark.data.transforms import LetterBox, ToTensorYOLO
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.structures.image_list import ImageList
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer


def ensure_detector(path, model_name):
    path = os.path.abspath(path)
    if os.path.exists(path):
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    downloaded = attempt_download_asset(model_name)
    downloaded = os.path.abspath(str(downloaded))
    if not os.path.exists(downloaded):
        raise FileNotFoundError(f"Ultralytics did not produce {model_name}: {downloaded}")
    if downloaded != path:
        shutil.copy2(downloaded, path)
    return path


def letterbox_meta(h, w, size=640):
    gain = min(size / float(h), size / float(w))
    new_w, new_h = int(round(w * gain)), int(round(h * gain))
    dw, dh = (size - new_w) / 2.0, (size - new_h) / 2.0
    left = int(round(dw - 0.1))
    top = int(round(dh - 0.1))
    return gain, left, top


def unletterbox_box(box, h, w, size=640):
    gain, left, top = letterbox_meta(h, w, size=size)
    x1, y1, x2, y2 = [float(v) for v in box]
    x1 = min(max((x1 - left) / gain, 0.0), float(w))
    y1 = min(max((y1 - top) / gain, 0.0), float(h))
    x2 = min(max((x2 - left) / gain, 0.0), float(w))
    y2 = min(max((y2 - top) / gain, 0.0), float(h))
    return [x1, y1, x2, y2]


def tensor_list(x):
    if x is None:
        return []
    if torch.is_tensor(x):
        return x.detach().cpu().tolist()
    return np.asarray(x).tolist()


def main():
    os.makedirs(args.output, exist_ok=True)
    detector = ensure_detector(args.detector, args.detector_name)
    if not os.path.exists(args.checkpoint):
        raise FileNotFoundError(f"REACT relation checkpoint not found: {args.checkpoint}")

    cfg = load_config_from_file(args.config)
    with open_dict(cfg):
        cfg.model.pretrained_detector_ckpt = detector
        cfg.output_dir = args.output
        cfg.test.ims_per_batch = max(1, int(args.batch_size))

    stats = get_dataset_statistics(cfg)
    obj_classes = list(stats["obj_classes"])
    rel_classes = list(stats["rel_classes"])
    with open_dict(cfg):
        cfg.model.roi_box_head.num_classes = len(obj_classes)
        cfg.model.roi_relation_head.num_classes = len(rel_classes)

    model = build_detection_model(cfg)
    model.to(cfg.model.device)
    checkpointer = DetectronCheckpointer(cfg, model, save_dir=args.output)
    checkpointer.load_backbone(detector)
    checkpointer.load(args.checkpoint)

    if "world" in str(cfg.model.backbone.type).lower():
        model.backbone.load_txt_feats(obj_classes[1:])
    model.eval()
    model.backbone.eval()
    model.roi_heads.eval()

    image_paths = sorted(
        p for p in Path(args.images).iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    )
    if not image_paths:
        raise RuntimeError(f"No images found in {args.images}")

    letterbox = LetterBox((640, 640), auto=False, stride=32)
    to_tensor = ToTensorYOLO(half=False)
    predictions = {}
    idx_to_files = []

    batch_size = max(1, int(args.batch_size))
    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_tensors, batch_meta = [], []
        for path in batch_paths:
            bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if bgr is None:
                raise RuntimeError(f"Cannot read image: {path}")
            h, w = bgr.shape[:2]
            lb, _ = letterbox(bgr, None)
            tensor, _ = to_tensor(lb, None)
            batch_tensors.append(tensor)
            batch_meta.append((h, w))

        tensors = torch.stack(batch_tensors, dim=0).to(cfg.model.device)
        image_list = ImageList(tensors, [(640, 640)] * len(batch_tensors))
        with torch.inference_mode():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(str(cfg.dtype) == "float16"),
            ):
                outputs = model(image_list)

        for path, (h, w), out in zip(batch_paths, batch_meta, outputs):
            out_index = len(idx_to_files)
            idx_to_files.append(str(path))

            boxes = tensor_list(out.get("boxes"))
            boxes = [unletterbox_box(b, h, w) for b in boxes]
            scores = [float(v) for v in tensor_list(out.get("pred_scores"))]

            labels = tensor_list(out.get("labels"))
            if not labels:
                labels = [int(v) + 1 for v in tensor_list(out.get("pred_labels"))]
            labels = [int(v) for v in labels]

            rel_pairs = tensor_list(out.get("rel_pair_idxs"))
            rel_labels, rel_scores = [], []
            rel_prob = out.get("pred_rel_scores")
            if torch.is_tensor(rel_prob) and rel_prob.ndim == 2 and rel_prob.shape[0] > 0:
                if rel_prob.shape[1] > 1:
                    score_t, label_t = rel_prob[:, 1:].max(dim=1)
                    label_t = label_t + 1
                else:
                    score_t, label_t = rel_prob.max(dim=1)
                rel_labels = [int(v) for v in label_t.detach().cpu().tolist()]
                rel_scores = [float(v) for v in score_t.detach().cpu().tolist()]
            elif out.get("pred_rel_labels") is not None:
                rel_labels = [int(v) for v in tensor_list(out.get("pred_rel_labels"))]
                raw_scores = tensor_list(out.get("rel_scores"))
                rel_scores = [float(v) for v in raw_scores] if raw_scores else [0.0] * len(rel_labels)

            n_rel = min(len(rel_pairs), len(rel_labels))
            rel_pairs = [[int(a), int(b)] for a, b in rel_pairs[:n_rel]]
            rel_labels = rel_labels[:n_rel]
            rel_scores = rel_scores[:n_rel]

            predictions[str(out_index)] = {
                "bbox": boxes,
                "bbox_labels": labels,
                "bbox_scores": scores,
                "rel_pairs": rel_pairs,
                "rel_labels": rel_labels,
                "rel_scores": rel_scores,
            }

    info = {
        "idx_to_files": idx_to_files,
        "ind_to_classes": obj_classes,
        "ind_to_predicates": rel_classes,
    }
    with open(os.path.join(args.output, "custom_prediction.json"), "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False)
    with open(os.path.join(args.output, "custom_data_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False)
    print(f"Wrote {len(predictions)} scene graphs to {args.output}")


if __name__ == "__main__":
    main()
""".strip()

def ensure_react_yoloworldv2_config():
    os.makedirs(os.path.dirname(REACT_CONFIG_PATH), exist_ok=True)
    with open(REACT_CONFIG_PATH, "w", encoding="utf-8") as file:
        file.write(REACT_YOLOWORLDV2_CONFIG + "\n")
    return REACT_CONFIG_PATH


def ensure_react_current_main_runner():
    os.makedirs(os.path.dirname(REACT_RUNNER_PATH), exist_ok=True)
    with open(REACT_RUNNER_PATH, "w", encoding="utf-8") as file:
        file.write(REACT_CURRENT_MAIN_RUNNER + "\n")
    os.chmod(REACT_RUNNER_PATH, 0o755)
    return REACT_RUNNER_PATH



def resolve_react_model_weight():
    global REACT_MODEL_WEIGHT
    if REACT_MODEL_WEIGHT and os.path.exists(REACT_MODEL_WEIGHT):
        return REACT_MODEL_WEIGHT

    patterns = [
        "best_model_epoch_*.pth",
        "best_model*.pth",
        "model_final.pth",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(os.path.join(REACT_RUN_DIR, "**", pattern), recursive=True))
    candidates = list(dict.fromkeys(candidates))
    if not candidates:
        raise FileNotFoundError(
            "No REACT + YOLO-WorldV2 relation checkpoint was found in "
            f"{REACT_RUN_DIR}. Train REACTPredictor with the generated "
            "YOLO-WorldV2 config and place the resulting best_model*.pth here."
        )

    config_files = []
    for pattern in ("*.yaml", "*.yml"):
        config_files.extend(
            glob.glob(os.path.join(REACT_RUN_DIR, "**", pattern), recursive=True)
        )
    if config_files:
        config_text = "\n".join(
            open(path, "r", encoding="utf-8", errors="ignore").read()
            for path in config_files
        ).lower()
        if "reactpredictor" not in config_text or "yoloworld" not in config_text:
            raise RuntimeError(
                "The checkpoint directory does not describe a "
                "REACTPredictor + YOLO-World model."
            )

    def checkpoint_key(path):
        match = re.search(r"epoch_(\d+)", os.path.basename(path))
        return (int(match.group(1)) if match else -1, os.path.getmtime(path))

    REACT_MODEL_WEIGHT = max(candidates, key=checkpoint_key)
    return REACT_MODEL_WEIGHT

def _lookup_label(label_map, index):
    if isinstance(label_map, dict):
        return label_map.get(str(index), label_map.get(index, str(index)))
    if isinstance(label_map, list) and 0 <= int(index) < len(label_map):
        return label_map[int(index)]
    return str(index)

def scene_graph_to_caption(graph):
    """Build the contextual caption from REACT textual triples."""
    triples = []
    seen = set()
    for relation in graph.get("relations", []):
        text = "{} {} {}".format(
            relation["subject_label"],
            relation["predicate"],
            relation["object_label"],
        ).strip()
        if text and text not in seen:
            seen.add(text)
            triples.append(text)
    return ". ".join(triples) + ("." if triples else "")

class ReactSceneGraphStore:
    """
    Reads SGG-Benchmark SGDet output and exposes one graph per image.
    If outputs are missing, it exports dataset images and invokes CURRENT
    SGG-Benchmark main in its isolated `.venv`, using native YOLO-WorldV2 +
    the original REACTPredictor through the modern Hydra/OmegaConf API.
    """
    def __init__(self):
        self.prediction_path = os.path.join(REACT_OUTPUT_DIR, "custom_prediction.json")
        self.info_path = os.path.join(REACT_OUTPUT_DIR, "custom_data_info.json")
        self.image_map_path = os.path.join(REACT_IMAGE_DIR, "image_id_map.json")
        self.graphs = None

    @staticmethod
    def _safe_filename(image_id):
        raw = canonical_id(image_id)
        prefix = re.sub(r"[^A-Za-z0-9_.-]+", "_", raw)[:50] or "image"
        digest = hashlib.sha1(raw.encode("utf-8")).hexdigest()[:12]
        return f"{prefix}_{digest}.jpg"

    def export_unique_images(self, dataset):
        os.makedirs(REACT_IMAGE_DIR, exist_ok=True)
        existing_map = {}
        if os.path.exists(self.image_map_path):
            with open(self.image_map_path, "r", encoding="utf-8") as file:
                existing_map = json.load(file)

        seen = set(existing_map.values())
        image_map = dict(existing_map)
        for item in tqdm(dataset, desc="Exporting unique images for REACT"):
            sample = get_sample_fields(item, include_image=True)
            image_id = canonical_id(sample["image_id"])
            if image_id in seen:
                continue
            filename = self._safe_filename(image_id)
            path = os.path.join(REACT_IMAGE_DIR, filename)
            if not os.path.exists(path):
                sample["image"].save(path, format="JPEG", quality=95)
            image_map[filename] = image_id
            seen.add(image_id)

        with open(self.image_map_path, "w", encoding="utf-8") as file:
            json.dump(image_map, file, ensure_ascii=False, indent=2)
        return image_map

    def _validate_runtime(self):
        required = {
            "SGG-Benchmark repository": REACT_REPO_DIR,
            "SGG-Benchmark Python": REACT_PYTHON,
        }
        missing = [
            name for name, path in required.items()
            if not os.path.exists(path)
        ]
        if missing:
            raise FileNotFoundError(
                "Missing REACT runtime components: " + ", ".join(missing)
            )
        return resolve_react_model_weight()

    def run_react(self):
        checkpoint = self._validate_runtime()
        ensure_react_yoloworldv2_config()
        ensure_react_current_main_runner()
        os.makedirs(REACT_OUTPUT_DIR, exist_ok=True)
        command = [
            REACT_PYTHON,
            REACT_RUNNER_PATH,
            "--repo", REACT_REPO_DIR,
            "--config", REACT_CONFIG_PATH,
            "--checkpoint", checkpoint,
            "--detector", REACT_YOLOWORLD_DETECTOR_CKPT,
            "--detector-name", REACT_YOLOWORLD_MODEL_NAME,
            "--images", REACT_IMAGE_DIR,
            "--output", REACT_OUTPUT_DIR,
            "--batch-size", str(REACT_TEST_BATCH_SIZE),
        ]
        subprocess.run(command, cwd=REACT_REPO_DIR, check=True)

    def ensure_ready(self, dataset):
        if os.path.exists(self.prediction_path) and os.path.exists(self.info_path):
            return
        self.export_unique_images(dataset)
        if REACT_AUTO_RUN:
            self.run_react()
        if not (os.path.exists(self.prediction_path) and os.path.exists(self.info_path)):
            raise RuntimeError("REACT scene-graph generation did not produce the expected cache files.")

    def load(self, dataset):
        self.ensure_ready(dataset)
        with open(self.prediction_path, "r", encoding="utf-8") as file:
            predictions = json.load(file)
        with open(self.info_path, "r", encoding="utf-8") as file:
            info = json.load(file)
        with open(self.image_map_path, "r", encoding="utf-8") as file:
            image_map = json.load(file)

        idx_to_files = info["idx_to_files"]
        class_map = info["ind_to_classes"]
        predicate_map = info["ind_to_predicates"]
        graphs = {}

        for idx, image_path in enumerate(idx_to_files):
            filename = os.path.basename(image_path)
            image_id = image_map.get(filename)
            if image_id is None:
                continue
            pred = predictions.get(str(idx), predictions.get(idx, {}))
            boxes = pred.get("bbox", [])
            labels = pred.get("bbox_labels", [])
            scores = pred.get("bbox_scores", [])
            entities = []
            for box_index, (box, label) in enumerate(zip(boxes, labels)):
                entities.append({
                    "index": box_index,
                    "label": clean_phrase(_lookup_label(class_map, label)),
                    "box": [float(v) for v in box],
                    "score": float(scores[box_index]) if box_index < len(scores) else 0.0,
                })

            relations = []
            rel_pairs = pred.get("rel_pairs", [])
            rel_labels = pred.get("rel_labels", [])
            rel_scores = pred.get("rel_scores", [])
            for rel_index, (pair, rel_label) in enumerate(zip(rel_pairs, rel_labels)):
                if len(pair) != 2:
                    continue
                subject_index, object_index = map(int, pair)
                if subject_index >= len(entities) or object_index >= len(entities):
                    continue
                relations.append({
                    "subject_index": subject_index,
                    "object_index": object_index,
                    "subject_label": entities[subject_index]["label"],
                    "object_label": entities[object_index]["label"],
                    "predicate": clean_phrase(_lookup_label(predicate_map, rel_label)),
                    "score": float(rel_scores[rel_index]) if rel_index < len(rel_scores) else 0.0,
                })
            graphs[canonical_id(image_id)] = {
                "entities": entities,
                "relations": relations,
            }
        self.graphs = graphs
        return self

    def get(self, image_id):
        if self.graphs is None:
            raise RuntimeError("ReactSceneGraphStore.load(dataset) must be called first.")
        key = canonical_id(image_id)
        if key not in self.graphs:
            raise KeyError(f"REACT graph not found for image_id={key}")
        return self.graphs[key]

class ConceptNetBridge:
    """MiniLM alignment from REACT visual entities to ConceptNet + 1-hop expansion."""
    def __init__(self, minilm, cache_path=CONCEPTNET_CACHE_FILE):
        self.minilm = minilm
        self.cache_path = cache_path
        self.session = requests.Session()
        self.cache = {"align": {}, "neighbors": {}}
        if os.path.exists(cache_path):
            try:
                with open(cache_path, "r", encoding="utf-8") as file:
                    loaded = json.load(file)
                if isinstance(loaded, dict):
                    self.cache.update(loaded)
            except Exception as error:
                warnings.warn(f"Could not read ConceptNet cache: {error}")

    @staticmethod
    def _slug(text):
        text = clean_phrase(text)
        text = re.sub(r"[^a-z0-9_ -]+", "", text)
        text = re.sub(r"[ _]+", "_", text).strip("_")
        return text

    @staticmethod
    def _label_from_uri(uri):
        parts = str(uri).split("/")
        if len(parts) >= 4 and parts[1] == "c":
            return clean_phrase(unquote(parts[3]).replace("_", " "))
        return clean_phrase(uri)

    def _get_json(self, url, params=None):
        try:
            response = self.session.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except Exception as error:
            if CONCEPTNET_STRICT:
                raise RuntimeError(f"ConceptNet request failed: {error}") from error
            warnings.warn(f"ConceptNet request failed; using direct concept only: {error}")
            return {}

    def _related_candidates(self, label):
        slug = self._slug(label)
        direct_uri = f"/c/en/{slug}" if slug else "/c/en/object"
        payload = self._get_json(
            f"{CONCEPTNET_API}/related{direct_uri}",
            params={"filter": "/c/en", "limit": CONCEPTNET_RELATED_LIMIT},
        )
        uris = [direct_uri]
        for item in payload.get("related", []):
            uri = item.get("@id")
            if isinstance(uri, str) and uri.startswith("/c/en/"):
                uris.append(uri)
        return list(dict.fromkeys(uris))

    def align_label(self, label):
        key = clean_phrase(label)
        cached = self.cache["align"].get(key)
        if cached:
            return cached
        uris = self._related_candidates(key)
        labels = [self._label_from_uri(uri) for uri in uris]
        query_vec = self.minilm.encode([key], normalize_embeddings=True)
        node_vecs = self.minilm.encode(labels, normalize_embeddings=True, batch_size=128)
        scores = (np.asarray(query_vec) @ np.asarray(node_vecs).T)[0]
        best = int(np.argmax(scores))
        aligned = {
            "uri": uris[best],
            "label": labels[best],
            "score": float(scores[best]),
        }
        self.cache["align"][key] = aligned
        return aligned

    def one_hop(self, uri):
        cached = self.cache["neighbors"].get(uri)
        if cached is not None:
            return cached
        payload = self._get_json(
            f"{CONCEPTNET_API}/query",
            params={"node": uri, "other": "/c/en", "limit": CONCEPTNET_NEIGHBOR_LIMIT},
        )
        neighbors = []
        for edge in payload.get("edges", []):
            start = edge.get("start", {}).get("@id", "")
            end = edge.get("end", {}).get("@id", "")
            other = end if start == uri else start
            if isinstance(other, str) and other.startswith("/c/en/"):
                neighbors.append({"uri": other, "label": self._label_from_uri(other)})
        neighbors = list({item["uri"]: item for item in neighbors}.values())
        self.cache["neighbors"][uri] = neighbors
        return neighbors

    def top2_for_question(self, question, scene_labels):
        candidates = {}
        for label in dedupe_keep_order(scene_labels):
            aligned = self.align_label(label)
            candidates[aligned["uri"]] = {"uri": aligned["uri"], "label": aligned["label"]}
            for neighbor in self.one_hop(aligned["uri"]):
                candidates[neighbor["uri"]] = neighbor
        if not candidates:
            return [], []
        items = list(candidates.values())
        q = self.minilm.encode([question], normalize_embeddings=True)
        embs = self.minilm.encode(
            [item["label"] for item in items], normalize_embeddings=True, batch_size=256
        )
        scores = (np.asarray(q) @ np.asarray(embs).T)[0]
        order = np.argsort(-scores)[:2]
        selected = [items[int(i)]["label"] for i in order]
        diagnostics = [
            {**items[int(i)], "question_similarity": float(scores[int(i)])}
            for i in order
        ]
        return dedupe_keep_order(selected), diagnostics

    def flush(self):
        os.makedirs(os.path.dirname(self.cache_path), exist_ok=True)
        with open(self.cache_path, "w", encoding="utf-8") as file:
            json.dump(self.cache, file, ensure_ascii=False)

class PaperMultiViewProcessor:
    def __init__(self, dataset, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.nlp = spacy.load(SPACY_MODEL_NAME)
        self.gliner = GLiNER.from_pretrained(GLINER_MODEL_ID).to(self.device)
        self.minilm = SentenceTransformer(MINILM_MODEL_ID, device="cpu")
        self.react = ReactSceneGraphStore().load(dataset)
        self.conceptnet = ConceptNetBridge(self.minilm)

    def _align_visual_concepts(self, concepts, graph, image):
        entities = graph.get("entities", [])
        if not concepts or not entities:
            return [image], []
        entity_labels = [entity["label"] for entity in entities]
        concept_emb = self.minilm.encode(concepts, normalize_embeddings=True)
        entity_emb = self.minilm.encode(entity_labels, normalize_embeddings=True)
        similarities = np.asarray(concept_emb) @ np.asarray(entity_emb).T

        matched = []
        chosen_entity_indices = set()
        for concept_index, concept in enumerate(concepts):
            entity_index = int(np.argmax(similarities[concept_index]))
            entity = entities[entity_index]
            matched.append({
                "concept": concept,
                "entity_index": entity_index,
                "entity_label": entity["label"],
                "similarity": float(similarities[concept_index, entity_index]),
            })
            chosen_entity_indices.add(entity_index)

        images = [image]
        width, height = image.size
        for entity_index in sorted(chosen_entity_indices):
            xmin, ymin, xmax, ymax = entities[entity_index]["box"]
            xmin = max(0, math.floor(xmin) - CROP_PADDING)
            ymin = max(0, math.floor(ymin) - CROP_PADDING)
            xmax = min(width, math.ceil(xmax) + CROP_PADDING)
            ymax = min(height, math.ceil(ymax) + CROP_PADDING)
            if xmax - xmin >= MIN_CROP_SIZE and ymax - ymin >= MIN_CROP_SIZE:
                images.append(image.crop((xmin, ymin, xmax, ymax)))
        return images, matched

    def generate_visual_context(self, samples):
        questions = [sample["question"] for sample in samples]
        visual_concepts_batch = batch_extract_visual_concepts(
            questions, self.gliner, self.nlp
        )
        outputs = []
        for sample, visual_concepts in tqdm(
            zip(samples, visual_concepts_batch),
            total=len(samples),
            desc="REACT multi-view formation",
            leave=False,
        ):
            graph = self.react.get(sample["image_id"])
            caption = scene_graph_to_caption(graph)
            visual_images, visual_alignment = self._align_visual_concepts(
                visual_concepts, graph, sample["image"]
            )
            ax = extract_question_entity_anchors(sample["question"], self.nlp)
            e_top2, cn_diagnostics = self.conceptnet.top2_for_question(
                sample["question"],
                [entity["label"] for entity in graph.get("entities", [])],
            )
            anchor_queries = dedupe_keep_order(ax + e_top2)
            context_queries = [sample["question"]]
            if caption:
                context_queries.extend([
                    caption,
                    f"{sample['question']} {caption}".strip(),
                ])
            outputs.append({
                **sample,
                "caption": caption,
                "scene_graph": graph,
                "scene_graph_triples": [
                    f"{r['subject_label']} {r['predicate']} {r['object_label']}"
                    for r in graph.get("relations", [])
                ],
                "visual_concepts": visual_concepts,
                "visual_alignment": visual_alignment,
                "visual_images": visual_images,
                "context_queries": context_queries,
                "question_entity_anchors": ax,
                "conceptnet_top2": e_top2,
                "conceptnet_diagnostics": cn_diagnostics,
                "anchor_queries": anchor_queries,
            })
        self.conceptnet.flush()
        return outputs


# 5. Multi-view retrieval and RRF

Encodes every query view with LongCLIP-L, searches the Wiki6M FAISS index (`RETRIEVAL_TOP_K_PER_QUERY = 120` per query), and fuses the ranked lists with reciprocal rank fusion (`RRF_K = 60`). Output: `raw_retrieval.jsonl` with the top 350 candidates per question.

In [ ]:
class Retriever:
    def __init__(self, chunks_dir, model_type="longclip_large", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_type = model_type
        self._load_model()

        self.loaded_chunks = []
        index_files = sorted(glob.glob(os.path.join(chunks_dir, "*.index")))
        if not index_files:
            raise FileNotFoundError(f"No FAISS index found in {chunks_dir}")

        for index_file in tqdm(index_files, desc="Loading FAISS indices"):
            index = faiss.read_index(index_file, faiss.IO_FLAG_MMAP)
            metadata_file = (
                index_file.replace("wikidata_", "metadata_")
                .replace(".index", ".json")
            )
            with open(metadata_file, "r", encoding="utf-8") as file:
                metadata = json.load(file)
            self.loaded_chunks.append((index, metadata))

    def _load_model(self):
        if self.model_type in {"longclip_base", "longclip_large"}:
            from model import longclip

            checkpoint = "longclip-L.pt"
            if not os.path.exists(checkpoint):
                os.system(
                    "wget https://huggingface.co/BeichenZhang/LongCLIP-L/"
                    "resolve/main/longclip-L.pt -O longclip-L.pt"
                )
            self.model, self.image_transform = longclip.load(
                checkpoint, device=self.device
            )
            self.tokenize = longclip.tokenize
        else:
            model_id = "openai/clip-vit-base-patch32"
            self.model = CLIPModel.from_pretrained(model_id).to(self.device)
            self.processor = CLIPProcessor.from_pretrained(model_id)
            self.tokenizer = CLIPTokenizer.from_pretrained(model_id)
        self.model.eval()

    def encode_images(self, images, batch_size=32):
        if not images:
            return np.empty((0, 0), dtype="float32")
        features = []
        with torch.inference_mode():
            for start in range(0, len(images), batch_size):
                batch = images[start : start + batch_size]
                if self.model_type in {"longclip_base", "longclip_large"}:
                    inputs = torch.stack(
                        [self.image_transform(image) for image in batch]
                    ).to(self.device)
                    batch_features = self.model.encode_image(inputs)
                else:
                    inputs = self.processor(
                        images=batch, return_tensors="pt", padding=True
                    ).to(self.device)
                    batch_features = self.model.get_image_features(**inputs)
                    if hasattr(batch_features, "image_embeds"):
                        batch_features = batch_features.image_embeds
                batch_features = F.normalize(batch_features, dim=-1)
                features.append(batch_features.cpu())
        return torch.cat(features).numpy().astype("float32")

    def encode_texts(self, texts, batch_size=32):
        if not texts:
            return np.empty((0, 0), dtype="float32")
        features = []
        with torch.inference_mode():
            for start in range(0, len(texts), batch_size):
                batch = texts[start : start + batch_size]
                if self.model_type in {"longclip_base", "longclip_large"}:
                    inputs = self.tokenize(batch, truncate=True).to(self.device)
                    batch_features = self.model.encode_text(inputs)
                else:
                    inputs = self.tokenizer(
                        batch,
                        padding=True,
                        truncation=True,
                        max_length=77,
                        return_tensors="pt",
                    ).to(self.device)
                    batch_features = self.model.get_text_features(**inputs)
                    if hasattr(batch_features, "text_embeds"):
                        batch_features = batch_features.text_embeds
                batch_features = F.normalize(batch_features, dim=-1)
                features.append(batch_features.cpu())
        return torch.cat(features).numpy().astype("float32")

    def search_queries(self, vectors, top_k=RETRIEVAL_TOP_K_PER_QUERY):
        """Return one independent ranked R_u per query view; never merge views here."""
        if len(vectors) == 0:
            return []
        total_queries = len(vectors)
        candidates = [{} for _ in range(total_queries)]

        for index, metadata in self.loaded_chunks:
            if index.d != vectors.shape[1]:
                if vectors.shape[1] < index.d:
                    raise ValueError(
                        f"Embedding dim {vectors.shape[1]} is smaller than FAISS dim {index.d}."
                    )
                current_vectors = vectors[:, : index.d]
            else:
                current_vectors = vectors

            for start in range(0, total_queries, 4096):
                distances, indices = index.search(
                    current_vectors[start : start + 4096], k=top_k
                )
                for local_index, (scores, db_indices) in enumerate(zip(distances, indices)):
                    query_index = start + local_index
                    for score, db_index in zip(scores, db_indices):
                        if db_index < 0 or db_index >= len(metadata):
                            continue
                        item = metadata[db_index]
                        wikidata_id = item.get("wikidata_id", item.get("id"))
                        if not wikidata_id:
                            continue
                        if "wikipedia_title" in item:
                            content = (
                                f"{item.get('wikipedia_title', '')}: "
                                f"{item.get('wikipedia_summary', '')}"
                            )
                        else:
                            content = f"{item.get('label', '')} is {item.get('desc', '')}"
                        previous = candidates[query_index].get(wikidata_id)
                        if previous is None or score > previous["score"]:
                            candidates[query_index][wikidata_id] = {
                                "wikidata_id": wikidata_id,
                                "content": content,
                                "score": float(score),
                            }

        ranked_lists = []
        for per_query in candidates:
            ranked = sorted(
                per_query.values(), key=lambda item: item["score"], reverse=True
            )[:top_k]
            ranked_lists.append(ranked)
        return ranked_lists

def split_ranked_lists(ranked_lists, counts):
    output = []
    cursor = 0
    for count in counts:
        output.append(ranked_lists[cursor : cursor + count])
        cursor += count
    if cursor != len(ranked_lists):
        raise ValueError("counts does not match the number of ranked lists.")
    return output

def reciprocal_rank_fusion_many(ranked_lists, k=RRF_K):
    """Eq. (4) of the paper: sum of 1 / (k_rrf + one-indexed rank) over every R_u."""
    fused = {}
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked, start=1):
            wikidata_id = item.get("wikidata_id", item.get("id"))
            if not wikidata_id:
                continue
            record = fused.setdefault(
                wikidata_id,
                {
                    "wikidata_id": wikidata_id,
                    "content": item["content"],
                    "score": float(item.get("score", 0.0)),
                    "rrf_score": 0.0,
                    "view_hits": 0,
                },
            )
            record["rrf_score"] += 1.0 / (k + rank)
            record["view_hits"] += 1
            record["score"] = max(record["score"], float(item.get("score", 0.0)))
    return sorted(fused.values(), key=lambda item: item["rrf_score"], reverse=True)


In [ ]:
def run_retrieval():
    dataset = load_vqa_dataset()
    processed_ids = read_processed_ids(RAW_KNOWLEDGE_FILE)
    pending_indices = [
        index
        for index, question_id in enumerate(dataset["question_id"])
        if canonical_id(question_id) not in processed_ids
    ]
    print(f"{len(pending_indices):,} samples remain for retrieval.")
    if not pending_indices:
        return

    retriever = Retriever(CHUNKS_DIR, MODEL_TYPE, device)
    multi_view = PaperMultiViewProcessor(dataset, device)

    with open(RAW_KNOWLEDGE_FILE, "a", encoding="utf-8") as output_file:
        for start in range(0, len(pending_indices), PIPELINE_BATCH_SIZE):
            indices = pending_indices[start : start + PIPELINE_BATCH_SIZE]
            samples = [get_sample_fields(dataset[index]) for index in indices]
            processed_samples = multi_view.generate_visual_context(samples)

            all_images, image_counts = [], []
            all_contexts, context_counts = [], []
            all_anchors, anchor_counts = [], []

            for sample in processed_samples:
                all_images.extend(sample["visual_images"])
                image_counts.append(len(sample["visual_images"]))

                all_contexts.extend(sample["context_queries"])
                context_counts.append(len(sample["context_queries"]))

                all_anchors.extend(sample["anchor_queries"])
                anchor_counts.append(len(sample["anchor_queries"]))

            image_ranked = retriever.search_queries(
                retriever.encode_images(all_images), RETRIEVAL_TOP_K_PER_QUERY
            )
            context_ranked = retriever.search_queries(
                retriever.encode_texts(all_contexts), RETRIEVAL_TOP_K_PER_QUERY
            )
            anchor_ranked = retriever.search_queries(
                retriever.encode_texts(all_anchors), RETRIEVAL_TOP_K_PER_QUERY
            ) if all_anchors else []

            image_by_sample = split_ranked_lists(image_ranked, image_counts)
            context_by_sample = split_ranked_lists(context_ranked, context_counts)
            anchor_by_sample = (
                split_ranked_lists(anchor_ranked, anchor_counts)
                if all_anchors
                else [[] for _ in processed_samples]
            )

            for sample, image_lists, context_lists, anchor_lists in zip(
                processed_samples, image_by_sample, context_by_sample, anchor_by_sample
            ):
                ranked_lists = image_lists + context_lists + anchor_lists
                fused = reciprocal_rank_fusion_many(ranked_lists, k=RRF_K)
                output_file.write(
                    json.dumps(
                        {
                            "question_id": sample["question_id"],
                            "image_id": sample["image_id"],
                            "question_text": sample["question"],
                            "caption": sample["caption"],
                            "context_queries": sample["context_queries"],
                            "visual_concepts": sample["visual_concepts"],
                            "visual_alignment": sample["visual_alignment"],
                            "question_entity_anchors": sample["question_entity_anchors"],
                            "conceptnet_top2": sample["conceptnet_top2"],
                            "anchor_queries": sample["anchor_queries"],
                            "scene_graph_triples": sample["scene_graph_triples"],
                            "num_query_views": len(ranked_lists),
                            "raw_candidates": fused[:RAW_FUSED_TOP_K],
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )
            output_file.flush()

            del (
                samples,
                processed_samples,
                all_images,
                all_contexts,
                all_anchors,
                image_ranked,
                context_ranked,
                anchor_ranked,
                image_by_sample,
                context_by_sample,
                anchor_by_sample,
            )
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del retriever, multi_view
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Retrieval cache saved to {RAW_KNOWLEDGE_FILE}")

run_retrieval()


# 6. Cross-encoder reranking

Scores the fused candidates with `mixedbread-ai/mxbai-rerank-large-v1` and keeps the top 50 (`cross_encoder_top50.jsonl`). When `Wiki6M_ver_1_0.jsonl.gz` is available, the full Wikipedia summary replaces the 300-character index text.

In [ ]:
class CrossEncoderReranker:
    def __init__(self, model_id=CROSS_ENCODER_MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CrossEncoder(
            model_id, device=self.device, trust_remote_code=True
        )

    def rerank(self, candidates, question, caption, top_k=RERANK_TOP_K):
        if not candidates:
            return []
        query = f"{question} Context: {caption}" if caption else question
        passages = [candidate["content"] for candidate in candidates]
        pairs = [[query, passage] for passage in passages]
        scores = np.asarray(
            self.model.predict(
                pairs,
                batch_size=128,
                show_progress_bar=False,
                convert_to_tensor=False,
            )
        ).reshape(-1)

        reranked = []
        for candidate, score in zip(candidates, scores):
            reranked.append(
                {
                    "content": candidate["content"],
                    "wikidata_id": candidate["wikidata_id"],
                    "rerank_score": float(score),
                    "origin_score": float(candidate.get("score", 0.0)),
                }
            )
        reranked.sort(key=lambda item: item["rerank_score"], reverse=True)
        return reranked[:top_k]

def load_wiki6m_dictionary(path):
    if not os.path.exists(path):
        print(f"{path} was not found; using the short FAISS metadata text.")
        return {}
    full_dictionary = {}
    with gzip.open(path, "rt", encoding="utf-8") as file:
        for line in tqdm(file, desc="Loading Wiki6M"):
            item = json.loads(line)
            wikidata_id = item.get("wikidata_id", item.get("id"))
            if wikidata_id:
                full_dictionary[wikidata_id] = (
                    f"{item.get('wikipedia_title', '')}: "
                    f"{item.get('wikipedia_summary', '')}"
                )
    return full_dictionary


In [ ]:
def run_cross_encoder_reranking():
    processed_ids = read_processed_ids(RERANKED_TOPM_FILE)
    reranker = CrossEncoderReranker()
    wiki6m = load_wiki6m_dictionary(WIKI6M_PATH)

    with open(RAW_KNOWLEDGE_FILE, "r", encoding="utf-8") as input_file, open(
        RERANKED_TOPM_FILE, "a", encoding="utf-8"
    ) as output_file:
        for line in tqdm(input_file, desc="Cross-encoder reranking"):
            data = json.loads(line)
            if canonical_id(data["question_id"]) in processed_ids:
                continue

            results = reranker.rerank(
                data.get("raw_candidates", []),
                data.get("question_text", ""),
                data.get("caption", ""),
            )
            knowledge = []
            for result in results:
                full_content = wiki6m.get(result["wikidata_id"], result["content"])
                result["content"] = full_content
                knowledge.append(full_content)

            output_file.write(
                json.dumps(
                    {
                        "question_id": data["question_id"],
                        "image_id": data["image_id"],
                        "question_text": data.get("question_text", ""),
                        "caption": data.get("caption", ""),
                        "retrieved_knowledge": knowledge,
                        "retrieved_metadata": results,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )
            output_file.flush()

    release_memory(reranker, wiki6m)
    print(f"Saved top-{RERANK_TOP_K} to {RERANKED_TOPM_FILE}")

run_cross_encoder_reranking()


## 6.1. Semantic-compressive evaluator (SCE)

The code calls this stage *Mamba gating*, but it does not apply the confidence gate. It combines the cross-encoder probability with the Mamba-2 representation-energy similarity `1 - D_str` (Eq. 9 of the paper, equal weights) on the top 25 cross-encoder candidates and keeps the top 10 (`mamba_top10.jsonl`).

In [ ]:
W_MXBAI = 0.5
W_MAMBA = 0.5

class MambaCompressorGate:
    def __init__(self, model_id=MAMBA_MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
        if self.tokenizer.eos_token_id is None:
            self.tokenizer.eos_token = self.tokenizer.pad_token
        self.pad_token_id = self.tokenizer.eos_token_id
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = MambaLMHeadModel.from_pretrained(
            model_id, device=self.device, dtype=dtype
        ).eval()

    def tokenize(self, text, max_length=512):
        return self.tokenizer.encode(
            text,
            return_tensors="pt",
            max_length=max_length,
            truncation=True,
        ).to(self.device)

    def get_batch_density(self, tensors):
        if isinstance(tensors, torch.Tensor):
            tensors = [tensors]
        if not tensors:
            return []

        lengths = torch.tensor(
            [tensor.shape[1] for tensor in tensors],
            device=self.device,
            dtype=torch.long,
        )
        max_length = int(lengths.max().item())
        if max_length % 8:
            max_length += 8 - max_length % 8

        padded = [
            F.pad(
                tensor,
                (0, max_length - tensor.shape[1]),
                value=self.pad_token_id,
            )
            for tensor in tensors
        ]
        batch = torch.cat(padded, dim=0)

        with torch.inference_mode():
            hidden = self.model.backbone(batch)
            token_energy = hidden.float().pow(2).mean(dim=2)
            mask = (
                torch.arange(max_length, device=self.device)[None, :]
                < lengths[:, None]
            )
            densities = (
                (token_energy * mask).sum(dim=1)
                / lengths.clamp_min(1).float()
            )
        return densities.cpu().tolist()

    def compute_similarity(self, query_tokens, passage_tokens):
        query_density = self.get_batch_density([query_tokens])[0]
        passage_densities = self.get_batch_density(passage_tokens)
        joint_tensors = [
            torch.cat([query_tokens, passage], dim=1)
            for passage in passage_tokens
        ]
        joint_densities = self.get_batch_density(joint_tensors)

        similarities = []
        for passage_density, joint_density in zip(
            passage_densities, joint_densities
        ):
            maximum = max(query_density, passage_density)
            minimum = min(query_density, passage_density)
            ncd = (joint_density - minimum) / maximum if maximum > 0 else 1.0
            similarities.append(1.0 - max(0.0, min(1.0, ncd)))
        return similarities

def sigmoid_scalar(value):
    value = max(-40.0, min(40.0, float(value)))
    return 1.0 / (1.0 + math.exp(-value))


In [ ]:
def run_mamba_gating():
    processed_ids = read_processed_ids(RERANKED_KNOWLEDGE)
    gate = MambaCompressorGate()

    with open(RERANKED_TOPM_FILE, "r", encoding="utf-8") as input_file, open(
        RERANKED_KNOWLEDGE, "a", encoding="utf-8"
    ) as output_file:
        for line in tqdm(input_file, desc="Mamba gating"):
            data = json.loads(line)
            if canonical_id(data["question_id"]) in processed_ids:
                continue

            candidates = data.get("retrieved_metadata", [])[:MAMBA_INPUT_TOP_M]
            if not candidates:
                continue

            query = (
                f"Question: {data.get('question_text', '')} "
                f"Context: {data.get('caption', '')} Answer:"
            )
            query_tokens = gate.tokenize(query)
            passage_tokens = [
                gate.tokenize(" " + candidate["content"])
                for candidate in candidates
            ]
            mamba_scores = gate.compute_similarity(query_tokens, passage_tokens)

            scored = []
            for candidate, mamba_score in zip(candidates, mamba_scores):
                mxbai_probability = sigmoid_scalar(
                    candidate.get("rerank_score", candidate.get("score", 0.0))
                )
                fusion_score = (
                    W_MXBAI * mxbai_probability + W_MAMBA * mamba_score
                )
                scored.append(
                    {
                        **candidate,
                        "fusion_score": float(fusion_score),
                        "breakdown": {
                            "mxbai_probability": float(mxbai_probability),
                            "mamba_similarity": float(mamba_score),
                        },
                    }
                )

            scored.sort(key=lambda item: item["fusion_score"], reverse=True)
            selected = scored[:FINAL_KNOWLEDGE_TOP_K]
            output_file.write(
                json.dumps(
                    {
                        "question_id": data["question_id"],
                        "image_id": data["image_id"],
                        "question_text": data.get("question_text", ""),
                        "caption": data.get("caption", ""),
                        "retrieved_knowledge": [
                            candidate["content"] for candidate in selected
                        ],
                        "retrieved_metadata": selected,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )
            output_file.flush()

    release_memory(gate)
    print(f"Saved top-{FINAL_KNOWLEDGE_TOP_K} to {RERANKED_KNOWLEDGE}")

run_mamba_gating()


# 7. Summary dataset

Joins the selected passages with the questions and reference answers (`summary_dataset.json`).

In [ ]:
def choose_training_label(answers, knowledge_texts):
    if not answers:
        return ""
    counts = Counter(answers)
    combined_knowledge = " ".join(knowledge_texts).lower()
    matched = []
    for answer in counts:
        clean = str(answer).strip().lower()
        if clean and re.search(
            r"(?<!\w)" + re.escape(clean) + r"(?!\w)", combined_knowledge
        ):
            matched.append(answer)
    return (
        max(matched, key=lambda answer: counts[answer])
        if matched
        else counts.most_common(1)[0][0]
    )

def build_summary_dataset():
    dataset = load_vqa_dataset()
    retrieval_map = {}
    with open(RERANKED_KNOWLEDGE, "r", encoding="utf-8") as file:
        for line in file:
            item = json.loads(line)
            retrieval_map[canonical_id(item["question_id"])] = item

    metadata_dataset = dataset.remove_columns(["image"])
    summary = []
    missing_knowledge = 0
    for raw_item in tqdm(metadata_dataset, desc="Building summary dataset"):
        sample = get_sample_fields(raw_item, include_image=False)
        retrieval = retrieval_map.get(canonical_id(sample["question_id"]), {})
        knowledge = retrieval.get("retrieved_knowledge", [])
        answers = sample["answers"]

        if DATASET_SPLIT == "train" and not answers:
            continue

        label = choose_training_label(answers, knowledge)
        if DATASET_SPLIT == "train":
            combined = " ".join(knowledge).lower()
            has_answer_match = bool(knowledge) and any(
                re.search(
                    r"(?<!\w)" + re.escape(answer.lower()) + r"(?!\w)",
                    combined,
                )
                for answer in answers
                if answer
            )
            if not has_answer_match:
                missing_knowledge += 1

        summary.append(
            {
                "question_id": sample["question_id"],
                "image_id": sample["image_id"],
                "question_text": sample["question"],
                "caption": retrieval.get("caption", ""),
                "knowledge_texts": knowledge,
                "label": label,
                "answers_list": answers,
            }
        )

    with open(SUMMARY_DATA, "w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False)

    if DATASET_SPLIT == "train" and summary:
        hit_rate = 100.0 * (1.0 - missing_knowledge / len(summary))
        print(f"Retrieval answer hit-rate: {hit_rate:.2f}%")
    print(f"Saved {len(summary):,} samples to {SUMMARY_DATA}")

build_summary_dataset()


# 8. Task-specific reranker: targets, training, and top-1 selection

In the code, the task-specific reranker of the paper is called the fine-tuned cross-encoder. A passage receives a hard label when it contains a reference answer. Questions without a hard positive receive soft targets `sigmoid(TEACHER_SCALE * z)` from the frozen teacher (Eq. 10). Targets and training run only on the `train` split. `build_top1_knowledge_dataset` then writes the selected passage and its score `S_max` for every question (`dataset_one_knowledge.json`).

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    for punctuation in string.punctuation:
        text = text.replace(punctuation, " ")
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split()).strip()

def check_answer_in_chunk(chunk, answers):
    normalized_chunk = normalize_text(chunk)
    for answer in parse_answer_list(answers):
        normalized_answer = normalize_text(answer)
        if normalized_answer and re.search(
            r"(?<!\w)" + re.escape(normalized_answer) + r"(?!\w)",
            normalized_chunk,
        ):
            return True
    return False

TEACHER_SCALE = 3.0  # omega in Eq. (10) of the paper

def create_cross_encoder_targets():
    with open(SUMMARY_DATA, "r", encoding="utf-8") as file:
        dataset = json.load(file)

    hard_match, soft_match = [], []
    for item in tqdm(dataset, desc="Creating hard labels"):
        labels = [
            1.0 if check_answer_in_chunk(chunk, item["answers_list"]) else 0.0
            for chunk in item.get("knowledge_texts", [])
        ]
        if any(labels):
            item["target_labels"] = labels
            item["label_type"] = "hard_match"
            hard_match.append(item)
        else:
            soft_match.append(item)

    if soft_match:
        reranker = CrossEncoder(
            CROSS_ENCODER_MODEL_ID, device=device, trust_remote_code=True
        )
        pairs = []
        counts = []
        for item in soft_match:
            query = (
                f"Context: {item.get('caption', '')}. "
                f"Question: {item.get('question_text', '')}. "
                f"Answer: {item.get('label', '')}"
            )
            knowledge = item.get("knowledge_texts", [])
            counts.append(len(knowledge))
            pairs.extend([[query, chunk] for chunk in knowledge])

        scores = (
            np.asarray(
                reranker.predict(
                    pairs,
                    batch_size=256,
                    show_progress_bar=True,
                    convert_to_tensor=False,
                )
            ).reshape(-1)
            if pairs
            else np.array([])
        )
        probabilities = torch.sigmoid(
            torch.tensor(scores, dtype=torch.float32) * TEACHER_SCALE
        ).tolist()

        cursor = 0
        for item, count in zip(soft_match, counts):
            item["target_labels"] = probabilities[cursor : cursor + count]
            item["label_type"] = "soft_cross_encoder"
            cursor += count
        assert cursor == len(probabilities)
        release_memory(reranker)

    combined = hard_match + soft_match
    random.Random(SEED).shuffle(combined)
    with open(MXBAI_LABEL, "w", encoding="utf-8") as file:
        json.dump(combined, file, ensure_ascii=False)
    print(f"Saved {len(combined):,} labeled samples to {MXBAI_LABEL}")

if DATASET_SPLIT == "train":
    create_cross_encoder_targets()
else:
    print("Skipping Cross-Encoder target creation outside the training split.")


In [ ]:
MAX_KNOWLEDGE_NODES = FINAL_KNOWLEDGE_TOP_K
MAX_TOTAL_LEN = 350
ce_tokenizer = AutoTokenizer.from_pretrained(CROSS_ENCODER_MODEL_ID)

class CrossEncoderDataset(Dataset):
    def __init__(self, json_path, tokenizer, limit=None):
        self.tokenizer = tokenizer
        with open(json_path, "r", encoding="utf-8") as file:
            data = json.load(file)
        self.data = data[:limit] if limit else data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        query = (
            f"Context: {item.get('caption', '')}. "
            f"Question: {item.get('question_text', '')}"
        )
        knowledge = list(item.get("knowledge_texts", []))[:MAX_KNOWLEDGE_NODES]
        labels = list(item.get("target_labels", []))[:MAX_KNOWLEDGE_NODES]
        node_mask = [1.0] * len(knowledge)

        while len(labels) < len(knowledge):
            labels.append(0.0)
        while len(knowledge) < MAX_KNOWLEDGE_NODES:
            knowledge.append("")
            labels.append(0.0)
            node_mask.append(0.0)

        encodings = self.tokenizer(
            [query] * MAX_KNOWLEDGE_NODES,
            knowledge,
            truncation=True,
            max_length=MAX_TOTAL_LEN,
            padding=False,
        )
        return {
            "input_ids": encodings["input_ids"],
            "attention_mask": encodings["attention_mask"],
            "labels": labels,
            "node_mask": node_mask,
        }

def collate_cross_encoder(batch):
    flat_ids = [
        torch.tensor(ids, dtype=torch.long)
        for item in batch
        for ids in item["input_ids"]
    ]
    flat_masks = [
        torch.tensor(mask, dtype=torch.long)
        for item in batch
        for mask in item["attention_mask"]
    ]
    input_ids = pad_sequence(
        flat_ids,
        batch_first=True,
        padding_value=ce_tokenizer.pad_token_id,
    )
    attention_mask = pad_sequence(
        flat_masks, batch_first=True, padding_value=0
    )
    batch_size = len(batch)
    return {
        "input_ids": input_ids.view(batch_size, MAX_KNOWLEDGE_NODES, -1),
        "attention_mask": attention_mask.view(
            batch_size, MAX_KNOWLEDGE_NODES, -1
        ),
        "labels": torch.tensor(
            [item["labels"] for item in batch], dtype=torch.float32
        ),
        "node_mask": torch.tensor(
            [item["node_mask"] for item in batch], dtype=torch.float32
        ),
    }

class FinetuneCrossEncoderScorer(nn.Module):
    def __init__(self, model_id):
        super().__init__()
        dtype = (
            torch.bfloat16
            if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
            else torch.float32
        )
        self.encoder = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            num_labels=1,
            torch_dtype=dtype,
            trust_remote_code=True,
        )
        self.loss_function = nn.BCEWithLogitsLoss(reduction="none")

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,
        node_mask=None,
        **kwargs,
    ):
        batch_size, node_count, sequence_length = input_ids.shape
        outputs = self.encoder(
            input_ids=input_ids.view(batch_size * node_count, sequence_length),
            attention_mask=attention_mask.view(
                batch_size * node_count, sequence_length
            ),
        )
        logits = outputs.logits.squeeze(-1).view(batch_size, node_count)
        loss = None
        if labels is not None:
            element_loss = self.loss_function(logits.float(), labels.float())
            if node_mask is None:
                loss = element_loss.mean()
            else:
                mask = node_mask.float()
                loss = (element_loss * mask).sum() / mask.sum().clamp_min(1.0)
        return {
            "loss": loss,
            "logits": logits,
            "probabilities": torch.sigmoid(logits.float()),
        }

    def gradient_checkpointing_enable(self, **kwargs):
        self.encoder.gradient_checkpointing_enable(**kwargs)
        embeddings = self.encoder.get_input_embeddings()
        if embeddings is not None:
            embeddings.requires_grad_(True)


In [ ]:
def train_cross_encoder():
    if DATASET_SPLIT != "train":
        raise ValueError("Cross-Encoder can only be fine-tuned on DATASET_SPLIT='train'.")

    train_dataset = CrossEncoderDataset(MXBAI_LABEL, ce_tokenizer)
    model = FinetuneCrossEncoderScorer(CROSS_ENCODER_MODEL_ID).to(device)

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    training_args = TrainingArguments(
        output_dir=MODEL_DIR,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
        num_train_epochs=8,
        learning_rate=1e-4,
        warmup_steps=50,
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=2,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        label_names=["labels"],
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collate_cross_encoder,
    )

    sample_batch = collate_cross_encoder(
        [train_dataset[0], train_dataset[min(1, len(train_dataset) - 1)]]
    )
    sample_batch = {key: value.to(device) for key, value in sample_batch.items()}
    with torch.no_grad():
        check = model(**sample_batch)
    print(f"Pre-flight loss: {check['loss'].item():.4f}")

    trainer.train()
    weights_path = os.path.join(MODEL_DIR, "cross_encoder_final_weights.pth")
    torch.save(model.state_dict(), weights_path)
    print(f"Cross-Encoder saved to {weights_path}")

if DATASET_SPLIT == "train":
    train_cross_encoder()
else:
    print("Skipping Cross-Encoder fine-tuning outside the training split.")


In [ ]:
def evaluate_cross_encoder(threshold=0.9):
    weights_path = os.path.join(MODEL_DIR, "cross_encoder_final_weights.pth")
    model = FinetuneCrossEncoderScorer(CROSS_ENCODER_MODEL_ID).to(device)
    model.load_state_dict(
        torch.load(weights_path, map_location=device), strict=True
    )
    model.eval()

    dataset = CrossEncoderDataset(MXBAI_LABEL, ce_tokenizer)
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_cross_encoder,
        num_workers=0,
    )
    predictions = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Evaluating Cross-Encoder"):
            batch = {key: value.to(device) for key, value in batch.items()}
            predictions.extend(model(**batch)["probabilities"].cpu().tolist())

    with open(MXBAI_LABEL, "r", encoding="utf-8") as file:
        original = json.load(file)

    valid_total = valid_top1 = invalid_total = invalid_false_positive = 0
    for item, probabilities in zip(original, predictions):
        valid_nodes = len(item.get("knowledge_texts", []))
        probabilities = probabilities[:valid_nodes]
        targets = item.get("target_labels", [])[:valid_nodes]
        if not probabilities:
            continue
        best_index = int(np.argmax(probabilities))
        max_probability = probabilities[best_index]
        if any(label > 0 for label in targets):
            valid_total += 1
            if max_probability >= threshold and targets[best_index] > 0:
                valid_top1 += 1
        else:
            invalid_total += 1
            if max_probability >= threshold:
                invalid_false_positive += 1

    print(
        f"Top-1 valid pass: {100 * valid_top1 / max(valid_total, 1):.2f}% | "
        f"False positive: "
        f"{100 * invalid_false_positive / max(invalid_total, 1):.2f}%"
    )
    release_memory(model)


In [ ]:
def build_top1_knowledge_dataset():
    weights_path = os.path.join(MODEL_DIR, "cross_encoder_final_weights.pth")
    model = FinetuneCrossEncoderScorer(CROSS_ENCODER_MODEL_ID).to(device)
    model.load_state_dict(
        torch.load(weights_path, map_location=device), strict=True
    )
    model.eval()

    dataset = CrossEncoderDataset(SUMMARY_DATA, ce_tokenizer)
    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_cross_encoder,
        num_workers=0,
    )
    predictions = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc="Selecting top-1 knowledge"):
            batch = {key: value.to(device) for key, value in batch.items()}
            predictions.extend(model(**batch)["probabilities"].cpu().tolist())

    with open(SUMMARY_DATA, "r", encoding="utf-8") as file:
        source = json.load(file)

    output = []
    for item, probabilities in zip(source, predictions):
        knowledge = list(item.get("knowledge_texts", []))
        if knowledge:
            valid_probabilities = probabilities[: len(knowledge)]
            best_index = int(np.argmax(valid_probabilities))
            selected_knowledge = knowledge[best_index]
            best_probability = float(valid_probabilities[best_index])
        else:
            selected_knowledge = ""
            best_probability = 0.0

        output.append(
            {
                "question_id": item["question_id"],
                "image_id": item.get("image_id", item["question_id"]),
                "question_text": item.get("question_text", ""),
                "caption": item.get("caption", ""),
                "knowledge_texts": selected_knowledge,
                "label": item.get("label", ""),
                "answers_list": item.get("answers_list", []),
                "best_knowledge_prob": best_probability,
            }
        )

    with open(SUMMARY_TOP_1, "w", encoding="utf-8") as file:
        json.dump(output, file, ensure_ascii=False)
    release_memory(model)
    print(f"Top-1 dataset saved to {SUMMARY_TOP_1}")

build_top1_knowledge_dataset()


# 9. Qwen2.5-VL LoRA fine-tuning (train split)

A training question receives its selected passage when `S_max >= TRAIN_PROB_THRESHOLD` (0.7). Checkpoints and the final adapter are pushed to `MODEL_REPO_ID`, and logs go to the private `TRACKING_REPO_ID`.

In [ ]:
TRAIN_PROB_THRESHOLD = 0.7
MAX_SEQ_LENGTH = 2048

if HF_USERNAME == "your-hf-username":
    raise ValueError("Set HF_USERNAME in 3. Configuration before training or evaluation.")

print("Logging in to Hugging Face Hub...")
login()


In [ ]:
class LossThresholdCallback(TrainerCallback):
    def __init__(self, threshold_loss, min_steps, divisor):
        self.threshold_loss = threshold_loss
        self.min_steps = min_steps
        self.divisor = divisor

    def on_log(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs=None,
        **kwargs,
    ):
        if not logs or "loss" not in logs:
            return
        if (
            state.global_step >= self.min_steps
            and state.global_step % self.divisor == 0
            and logs["loss"] <= self.threshold_loss
        ):
            control.should_training_stop = True

class TrainingLoggerCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.log_file = os.path.join(output_dir, "train_loss.jsonl")

    def on_log(
        self,
        args: TrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs=None,
        **kwargs,
    ):
        if logs and "loss" in logs:
            with open(self.log_file, "a", encoding="utf-8") as file:
                file.write(
                    json.dumps(
                        {
                            "step": state.global_step,
                            "epoch": state.epoch or 0,
                            "loss": logs["loss"],
                            "learning_rate": logs.get("learning_rate", 0),
                        }
                    )
                    + "\n"
                )

class HybridVQADataset(Dataset):
    def __init__(self, json_path, dataset_lookup, limit=None):
        self.lookup = dataset_lookup
        with open(json_path, "r", encoding="utf-8") as file:
            data = json.load(file)
        self.data = data[:limit] if limit else data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        raw_image = self.lookup.get_image(item["question_id"])
        question = str(item.get("question_text", "")).strip()
        answer = str(item.get("label", "")).strip()
        knowledge = item.get("knowledge_texts", "")
        if isinstance(knowledge, list):
            knowledge = " ".join(knowledge)
        knowledge = str(knowledge).replace("\n", " ").strip()
        probability = float(item.get("best_knowledge_prob", 0.0))

        prompt = (
            f"context: {knowledge}\nquestion: {question}"
            if knowledge and probability >= TRAIN_PROB_THRESHOLD
            else f"question: {question}"
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": raw_image},
                    {"type": "text", "text": prompt},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ]
        return {"messages": messages}

def find_last_subsequence(sequence, subsequence):
    last_index = -1
    width = len(subsequence)
    for index in range(len(sequence) - width + 1):
        if sequence[index : index + width] == subsequence:
            last_index = index
    return last_index

@dataclass
class QwenDataCollator:
    processor: Any
    max_length: int = MAX_SEQ_LENGTH

    def __post_init__(self):
        self.assistant_prefix = self.processor.tokenizer.encode(
            "<|im_start|>assistant\n", add_special_tokens=False
        )

    def __call__(self, features):
        messages = [feature["messages"] for feature in features]
        texts = [
            self.processor.apply_chat_template(
                message, tokenize=False, add_generation_prompt=False
            )
            for message in messages
        ]
        image_inputs, video_inputs = process_vision_info(messages)
        batch = self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding="longest",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        for row_index in range(labels.shape[0]):
            row = batch["input_ids"][row_index].tolist()
            start = find_last_subsequence(row, self.assistant_prefix)
            if start < 0:
                raise ValueError("Assistant prefix was not found in the Qwen prompt.")
            labels[row_index, : start + len(self.assistant_prefix)] = -100
        batch["labels"] = labels
        return batch


In [ ]:
def train_qwen():
    if DATASET_SPLIT != "train":
        raise ValueError("Qwen can only be fine-tuned on DATASET_SPLIT='train'.")

    api = HfApi()
    resume_path = None
    try:
        repo_files = api.list_repo_files(MODEL_REPO_ID, repo_type="model")
        checkpoint_folders = {
            path.split("/")[0]
            for path in repo_files
            if path.startswith("checkpoint-")
        }
        if checkpoint_folders:
            latest = max(
                checkpoint_folders,
                key=lambda value: int(value.replace("checkpoint-", "")),
            )
            snapshot_download(
                repo_id=MODEL_REPO_ID,
                allow_patterns=f"{latest}/*",
                local_dir=TEMP_MODEL_DIR,
            )
            resume_path = os.path.join(TEMP_MODEL_DIR, latest)
    except Exception:
        resume_path = None

    processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model_dtype = (
        torch.bfloat16
        if use_bf16
        else torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype=model_dtype,
        device_map="auto",
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    for parameter in model.parameters():
        parameter.requires_grad = False
    for name, parameter in model.named_parameters():
        if any(key in name.lower() for key in ("visual", "merger", "projector")):
            parameter.requires_grad = True

    model = get_peft_model(
        model,
        LoraConfig(
            r=32,
            lora_alpha=64,
            lora_dropout=0.2,
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj",
            ],
            task_type="CAUSAL_LM",
            inference_mode=False,
        ),
    )
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    model.print_trainable_parameters()

    train_dataset = HybridVQADataset(
        SUMMARY_TOP_1, get_dataset_lookup()
    )
    collator = QwenDataCollator(processor)

    mini_batch_size = 8
    accumulation = 4
    epochs = 6
    save_steps = 100
    steps_per_epoch = math.ceil(
        len(train_dataset) / (mini_batch_size * accumulation)
    )
    total_steps = steps_per_epoch * epochs

    training_args = TrainingArguments(
        output_dir=TEMP_MODEL_DIR,
        per_device_train_batch_size=mini_batch_size,
        gradient_accumulation_steps=accumulation,
        gradient_checkpointing=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        num_train_epochs=epochs,
        learning_rate=5e-6,
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
        lr_scheduler_type="cosine",
        warmup_steps=int(total_steps * 0.10),
        weight_decay=0.05,
        logging_steps=10,
        save_strategy="steps",
        save_steps=save_steps,
        save_total_limit=4,
        remove_unused_columns=False,
        report_to="tensorboard",
        logging_dir=os.path.join(TEMP_LOG_DIR, "tensorboard_runs"),
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator,
        callbacks=[
            LossThresholdCallback(0.3, total_steps // 2, save_steps),
            TrainingLoggerCallback(TEMP_LOG_DIR),
        ],
    )

    train_result = trainer.train(resume_from_checkpoint=resume_path)
    trainer.save_model(TEMP_MODEL_DIR)
    processor.save_pretrained(TEMP_MODEL_DIR)

    metrics = {
        "dataset": DATASET_NAME,
        "model_id": QWEN_MODEL_ID,
        "run_name": RUN_NAME,
        "total_steps": trainer.state.global_step,
        **train_result.metrics,
    }
    with open(
        os.path.join(TEMP_LOG_DIR, "train_summary_metadata.json"),
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(metrics, file, ensure_ascii=False, indent=2)

    api.create_repo(MODEL_REPO_ID, private=False, exist_ok=True)
    api.upload_folder(
        folder_path=TEMP_MODEL_DIR,
        repo_id=MODEL_REPO_ID,
        path_in_repo=".",
        commit_message=f"Upload model weights for {RUN_NAME}",
    )
    api.create_repo(TRACKING_REPO_ID, private=True, exist_ok=True)
    api.upload_folder(
        folder_path=TEMP_LOG_DIR,
        repo_id=TRACKING_REPO_ID,
        path_in_repo=RUN_NAME,
        commit_message=f"Add logs for {RUN_NAME}",
    )

if DATASET_SPLIT == "train":
    train_qwen()
else:
    print("Skipping Qwen fine-tuning outside the training split.")


# 10. Gated inference and evaluation (validation and test)

The gate passes the selected passage when `S_max >= threshold` (Eq. 12). On `validation`, every saved checkpoint is evaluated for thresholds 0.55, 0.7, 0.8, 0.9, and 1.0. On `test`, the final adapter is evaluated with threshold 0.9. Metrics are appended to `eval_results_tracker.jsonl`, and per-question predictions are saved as `predictions_<strategy>.json`.

In [ ]:
def available_eval_checkpoints(repo_id):
    try:
        files = HfApi().list_repo_files(repo_id, repo_type="model")
    except Exception:
        return [None]
    folders = {
        path.split("/")[0]
        for path in files
        if path.startswith("checkpoint-") and "/" in path
    }
    checkpoints = sorted(
        folders,
        key=lambda value: int(value.replace("checkpoint-", "")),
    )
    return checkpoints + [None]

if DATASET_SPLIT == "validation":
    EVAL_CHECKPOINTS = available_eval_checkpoints(MODEL_REPO_ID)
    EVAL_THRESHOLDS = [0.55, 0.7, 0.8, 0.9, 1.0]
elif DATASET_SPLIT == "test":
    EVAL_CHECKPOINTS = [None]
    EVAL_THRESHOLDS = [0.9]
else:
    EVAL_CHECKPOINTS = []
    EVAL_THRESHOLDS = []

EVAL_BATCH_SIZE = 32

test_plan = [
    {
        "strategy_name": (
            f"{DATASET_SLUG}_{checkpoint or 'final'}_rag_{threshold}"
        ),
        "checkpoint": checkpoint,
        "threshold": threshold,
        "batch_size": EVAL_BATCH_SIZE,
    }
    for checkpoint in EVAL_CHECKPOINTS
    for threshold in EVAL_THRESHOLDS
]


In [ ]:
def vqa_normalize_text(text):
    text = re.sub(r"[^\w\s]", "", str(text).lower())
    text = re.sub(r"\b(a|an|the)\b", "", text)
    return " ".join(text.split())

def compute_vqa_accuracy(predicted_answer, ground_truths):
    if not ground_truths:
        return 0.0
    prediction = vqa_normalize_text(predicted_answer)
    counts = Counter(vqa_normalize_text(answer) for answer in ground_truths)
    return min(counts.get(prediction, 0) / 3.0, 1.0)

class HybridEvalDataset(Dataset):
    def __init__(self, json_path, dataset_lookup, threshold, limit=None):
        self.lookup = dataset_lookup
        self.threshold = threshold
        with open(json_path, "r", encoding="utf-8") as file:
            data = json.load(file)
        self.data = data[:limit] if limit else data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        image = self.lookup.get_image(item["question_id"])
        question = str(item.get("question_text", "")).strip()
        knowledge = item.get("knowledge_texts", "")
        if isinstance(knowledge, list):
            knowledge = " ".join(knowledge)
        knowledge = str(knowledge).replace("\n", " ").strip()
        probability = float(item.get("best_knowledge_prob", 0.0))
        prompt = (
            f"context: {knowledge}\nquestion: {question}"
            if knowledge and probability >= self.threshold
            else f"question: {question}"
        )
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            "question_id": item["question_id"],
            "image_id": item.get("image_id", item["question_id"]),
            "question_text": question,
            "full_context": knowledge,
            "answers_list": item.get("answers_list", []),
        }

@dataclass
class QwenEvalDataCollator:
    processor: Any

    def __call__(self, features):
        messages = [feature["messages"] for feature in features]
        self.processor.tokenizer.padding_side = "left"
        texts = [
            self.processor.apply_chat_template(
                message, tokenize=False, add_generation_prompt=True
            )
            for message in messages
        ]
        image_inputs, video_inputs = process_vision_info(messages)
        batch = self.processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding="longest",
            return_tensors="pt",
        )
        batch["metadata"] = [
            {
                key: feature[key]
                for key in (
                    "question_id", "image_id", "question_text",
                    "full_context", "answers_list",
                )
            }
            for feature in features
        ]
        return batch

def load_hf_model_for_inference(repo_id, subfolder=None, device="cuda"):
    processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    processor.tokenizer.padding_side = "left"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model_dtype = (
        torch.bfloat16
        if use_bf16
        else torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype=model_dtype,
        device_map={"": device},
        attn_implementation="sdpa",
    )
    kwargs = {"subfolder": subfolder} if subfolder else {}
    model = PeftModel.from_pretrained(base_model, repo_id, **kwargs)
    return model.eval(), processor


In [ ]:
def evaluate_kg_vqa(
    model,
    processor,
    output_file,
    strategy_name,
    threshold,
    checkpoint_name,
    batch_size,
):
    dataset = HybridEvalDataset(
        SUMMARY_TOP_1, get_dataset_lookup(), threshold
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=QwenEvalDataCollator(processor),
        num_workers=0,
    )

    total_score = 0.0
    total_samples = 0
    total_generation_time = 0.0
    results = []
    start_time = time.time()

    with torch.inference_mode():
        for batch in tqdm(loader, desc=strategy_name):
            metadata = batch.pop("metadata")
            inputs = {key: value.to(device) for key, value in batch.items()}

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            generation_start = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
            )
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            total_generation_time += time.time() - generation_start

            input_length = inputs["input_ids"].shape[1]
            for output, item in zip(outputs, metadata):
                prediction = processor.tokenizer.decode(
                    output[input_length:], skip_special_tokens=True
                ).strip()
                score = compute_vqa_accuracy(
                    prediction, item["answers_list"]
                )
                total_score += score
                total_samples += 1
                results.append(
                    {
                        **item,
                        "extracted_answer": prediction,
                        "accuracy_score": score,
                    }
                )

    total_time = time.time() - start_time
    accuracy = 100.0 * total_score / max(total_samples, 1)
    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False)

    return {
        "dataset": DATASET_NAME,
        "split": DATASET_SPLIT,
        "strategy_name": strategy_name,
        "checkpoint_used": checkpoint_name or "final_model",
        "test_prob_threshold": threshold,
        "dataset_size": total_samples,
        "eval_batch_size": batch_size,
        "accuracy_vqa": round(accuracy, 2),
        "avg_llm_gen_sec": round(
            total_generation_time / max(total_samples, 1), 4
        ),
        "avg_e2e_latency_sec": round(
            total_time / max(total_samples, 1), 4
        ),
        "throughput_samples_per_sec": round(
            total_samples / max(total_time, 1e-9), 2
        ),
        "total_eval_time_mins": round(total_time / 60, 2),
    }

def run_evaluation_plan():
    api = HfApi()
    current_checkpoint = object()
    model = processor = None

    for plan in test_plan:
        if plan["checkpoint"] != current_checkpoint:
            if model is not None:
                del model, processor
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            model, processor = load_hf_model_for_inference(
                MODEL_REPO_ID, plan["checkpoint"], device
            )
            current_checkpoint = plan["checkpoint"]

        detailed_file = os.path.join(
            TEMP_LOG_DIR, f"predictions_{plan['strategy_name']}.json"
        )
        metrics = evaluate_kg_vqa(
            model,
            processor,
            detailed_file,
            plan["strategy_name"],
            plan["threshold"],
            plan["checkpoint"],
            plan["batch_size"],
        )
        with open(TRACKER_FILE, "a", encoding="utf-8") as file:
            file.write(json.dumps(metrics, ensure_ascii=False) + "\n")

        for attempt in range(1, 6):
            try:
                api.create_repo(
                    TRACKING_REPO_ID, private=True, exist_ok=True
                )
                api.upload_folder(
                    folder_path=TEMP_LOG_DIR,
                    repo_id=TRACKING_REPO_ID,
                    path_in_repo=RUN_NAME,
                    commit_message=(
                        f"Evaluation {plan['strategy_name']}"
                    ),
                )
                break
            except Exception as error:
                if attempt == 5:
                    raise
                time.sleep(15 * attempt)

if DATASET_SPLIT != "train":
    run_evaluation_plan()
else:
    print("Skipping evaluation on the training split.")
